# Section 01 lab — Spark foundations

Run the section setup once. Work through each activity from Step 1 onward; every step explains why it is needed before the code runs.


## Section setup — run once

Create one local Spark session and reuse it throughout this section.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Create a DataFrame from Python rows

Begin with visible Python data so the relationship between rows, columns and a DataFrame is clear.


### Step 1 — Define the rows and column names

The input is small enough to inspect before Spark receives it.


In [ ]:
people = [(1, 'Asha', 'Pune'), (2, 'Bilal', 'Mumbai'), (3, 'Chen', 'Pune')]
columns = ['person_id', 'name', 'city']
people


### Step 2 — Create the DataFrame

Spark converts the Python rows into a distributed table with named columns.


In [ ]:
people_df = spark.createDataFrame(people, columns)
people_df


### Step 3 — Inspect rows and types

Always verify the structure before applying transformations.


In [ ]:
people_df.printSchema()
people_df.show(truncate=False)
assert people_df.count() == 3


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
people_df.createOrReplaceTempView('section01_people')
sql_result = spark.sql('SELECT person_id, name, city FROM section01_people ORDER BY person_id')
sql_result.show()
assert sql_result.count() == people_df.count()


## Exercise 2 — Inspect DataFrame metadata

Learn which properties can be checked without introducing new transformations.


### Step 1 — Create an orders DataFrame

This exercise starts with its own input.


In [ ]:
orders = spark.createDataFrame([('O1', 500.0), ('O2', 750.0), ('O3', 1200.0)], ['order_id', 'amount'])


### Step 2 — Inspect columns and schema

Column names and data types form the DataFrame contract.


In [ ]:
print('Columns:', orders.columns)
orders.printSchema()


### Step 3 — Trigger a row count

count is an action because Spark must produce a result.


In [ ]:
print('Rows:', orders.count())
assert orders.count() == 3


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
orders.createOrReplaceTempView('section01_orders')
spark.sql('DESCRIBE section01_orders').show(truncate=False)
sql_result = spark.sql('SELECT COUNT(*) AS rows, SUM(amount) AS total_amount FROM section01_orders')
sql_result.show()


## Exercise 3 — Separate transformations from actions

Build a plan first, then observe which call requires Spark to execute it.


### Step 1 — Create a numeric DataFrame

spark.range creates one column named id.


In [ ]:
numbers = spark.range(0, 20)
numbers.printSchema()


### Step 2 — Describe a filtered result

filter returns another DataFrame; it does not immediately return rows.


In [ ]:
even_numbers = numbers.filter('id % 2 = 0')
even_numbers


### Step 3 — Run actions

show and count require actual results.


In [ ]:
even_numbers.show()
print('Even rows:', even_numbers.count())
assert even_numbers.count() == 10


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
numbers.createOrReplaceTempView('section01_numbers')
sql_result = spark.sql('SELECT id FROM section01_numbers WHERE id % 2 = 0 ORDER BY id')
sql_result.show()
assert sql_result.count() == even_numbers.count()


## Exercise 4 — Observe partitions

Connect the number of partitions to the number of tasks Spark can schedule for a stage.


### Step 1 — Create four starting partitions

The final range argument controls the initial partition count.


In [ ]:
numbers = spark.range(0, 16, 1, 4)
print('Partitions:', numbers.rdd.getNumPartitions())


### Step 2 — Display each row with its partition ID

spark_partition_id exposes the current placement of each row.


In [ ]:
from pyspark.sql import functions as F
located = numbers.withColumn('partition_id', F.spark_partition_id())
located.show(16)


### Step 3 — Reduce the partition count

coalesce reduces partitions without requesting a full redistribution.


In [ ]:
reduced = numbers.coalesce(2)
print('After coalesce:', reduced.rdd.getNumPartitions())
assert reduced.rdd.getNumPartitions() == 2


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
numbers.createOrReplaceTempView('section01_partition_source')
sql_result = spark.sql('SELECT /*+ REPARTITION(2) */ id FROM section01_partition_source')
print('SQL result partitions:', sql_result.rdd.getNumPartitions())
sql_result.show()


## Exercise 5 — Read a simple execution plan

Use a familiar filter before looking at the plan Spark intends to run.


### Step 1 — Create a filtered DataFrame

The expression is still lazy.


In [ ]:
numbers = spark.range(0, 100)
large_ids = numbers.filter('id >= 90')


### Step 2 — Display the formatted plan

Look for the range source followed by the filter.


In [ ]:
large_ids.explain('formatted')


### Step 3 — Execute and verify

The action now runs the plan.


In [ ]:
large_ids.show()
assert large_ids.count() == 10


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
numbers.createOrReplaceTempView('section01_plan_source')
spark.sql('EXPLAIN FORMATTED SELECT id FROM section01_plan_source WHERE id >= 90').show(truncate=False)
sql_result = spark.sql('SELECT id FROM section01_plan_source WHERE id >= 90 ORDER BY id')
sql_result.show()
assert sql_result.count() == large_ids.count()
